## 月度数据归档

对 landing、clean、c-dataset 三类源中超过 `retention_days` 的旧数据，按配置选择"归档+删除"或"仅删除"。

- `enable_archive=true`（默认）：先备份到 Blob 归档目录，再从源表删除。
- `enable_archive=false`：跳过归档，直接从源表删除。

### Widgets
- `archive_config_json`：JSON 数组，每个元素是一组归档配置。
- `archive_root`：Blob 归档根路径（可选，默认 `wasb://raw@saapseauatcepadl01.blob.core.windows.net/mdm_archive`）。
- `trigger_timestamp_ms`：触发时间毫秒戳（可选）。
- `dry_run`：设置为 `true` 时只统计匹配行数，不写入归档也不删除（可选，默认 `false`）。

```json
[
  {
    "group_id": "landing_consumer",
    "source_type": "path",
    "source": "abfss://bronze@saapseauatacdgen2.dfs.core.windows.net/mdm/mdm_raw_landing/consumerlist_raw_20260228",
    "time_col": "slndc_creation_dt",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "landing/consumer"
  },
  {
    "group_id": "clean_consumer",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.t_clean_consumer",
    "time_col": "SRCC_CREATION_DT",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "clean/consumer"
  },
  {
    "group_id": "c_cbr_dataset",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.c_cbr_dataset",
    "time_col": "_create_time",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "dataset/cbr"
  },
  {
    "group_id": "c_cbr_withoutpii_dataset",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.c_cbr_withoutpii_dataset",
    "time_col": "_create_time",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "dataset/cbr_withoutpii"
  },
  {
    "group_id": "c_cbrdrjart_dataset",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.c_cbrdrjart_dataset",
    "time_col": "_create_time",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "dataset/cbrdrjart"
  },
  {
    "group_id": "c_transaction_master_dataset",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.c_transaction_master_dataset",
    "time_col": "_create_time",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "dataset/transaction_master"
  },
  {
    "group_id": "c_membership_mapping_log",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.c_membership_mapping_log",
    "time_col": "_create_time",
    "retention_days": 365,
    "enable_archive": true,
    "archive_blob_prefix": "dataset/membership_mapping_log"
  },
  {
    "group_id": "clean_cid_edge",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.t_clean_cid_edge",
    "time_col": "SRCC_CREATION_DT",
    "retention_days": 365,
    "enable_archive": false
  },
  {
    "group_id": "clean_cid_group",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.t_clean_cid_group",
    "time_col": "SRCC_CREATION_DT",
    "retention_days": 365,
    "enable_archive": false
  },
  {
    "group_id": "clean_ukey_edge",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.t_clean_ukey_edge",
    "time_col": "SRCC_CREATION_DT",
    "retention_days": 365,
    "enable_archive": false
  },
  {
    "group_id": "clean_ukey_group",
    "source_type": "table",
    "source": "catalog_southeastasia_mdm_silver_uat.talend_uat_database.t_clean_ukey_group",
    "time_col": "SRCC_CREATION_DT",
    "retention_days": 365,
    "enable_archive": false
  }
]
```

### Workflow 配置
创建一个 Databricks Workflow 作业，添加一个 Notebook 任务指向本 Notebook。调度 cron 建议 `0 2 1 * * ?`（每月 1 日凌晨 2 点）。

In [0]:
%run ../00_common/data_utils

In [0]:
# 标准库和第三方库导入
import json
from datetime import datetime, timedelta, timezone
from typing import Any
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark import StorageLevel

In [0]:
# 默认保留天数，可通过配置按表覆盖
DEFAULT_RETENTION_DAYS = 365

# 默认 Blob 归档根目录，可通过 archive_root widget 覆盖
# 注意：实际存储账号需要与 env.py 中 sap_touchpoint_blob_config 保持一致
DEFAULT_ARCHIVE_ROOT = "wasb://raw@saapseauatcepadl01.blob.core.windows.net/mdm_archive"

In [0]:
def parse_archive_config(config_json: str) -> list[dict[str, Any]]:
    """
    解析并校验归档配置 JSON。

    参数:
        config_json: archive_config_json widget 传入的 JSON 字符串。

    返回:
        校验后的配置列表，每个元素包含:
        - group_id: 归档组唯一标识
        - source_type: "path" 或 "table"
        - source: Delta 路径或表名
        - time_col: 用于判断数据新旧的时间列
        - retention_days: 保留天数（默认 365）
        - enable_archive: 是否先归档再删除（默认 true）；false 表示仅删除不归档
        - archive_blob_prefix: Blob 归档目录前缀（enable_archive=true 时必填）

    每个配置项预期结构：
    {
        "group_id": "landing_consumer",
        "source_type": "path",
        "source": "abfss://bronze@.../consumerlist_raw",
        "time_col": "slndc_creation_dt",
        "retention_days": 365,
        "enable_archive": true,
        "archive_blob_prefix": "landing/consumer"
    }
    """
    # 空配置直接报错，避免误删全表
    if not config_json or not config_json.strip():
        raise ValueError("archive_config_json widget is required")

    # 解析为 Python 对象
    config = json.loads(config_json)
    if not isinstance(config, list) or len(config) == 0:
        raise ValueError("archive_config_json must be a non-empty list")

    validated = []
    # 必填字段集合（enable_archive=true 时才需要 archive_blob_prefix）
    required_keys = {"group_id", "source_type", "source", "time_col"}

    for idx, item in enumerate(config):
        # 检查必填字段是否缺失
        missing = required_keys - set(item.keys())
        if missing:
            raise ValueError(f"Item {idx} missing required fields: {missing}")

        # source_type 只能是 path（Delta 路径）或 table（注册表）
        if item["source_type"] not in {"path", "table"}:
            raise ValueError(f"Item {idx}: source_type must be 'path' or 'table'")

        enable_archive = bool(item.get("enable_archive", True))
        archive_blob_prefix = str(item.get("archive_blob_prefix", "")).strip("/")

        # 需要归档时，归档前缀不能为空
        if enable_archive and not archive_blob_prefix:
            raise ValueError(f"Item {idx}: archive_blob_prefix is required when enable_archive=true")

        # 统一转换为标准类型，避免后续类型不一致
        validated.append({
            "group_id": str(item["group_id"]),
            "source_type": str(item["source_type"]),
            "source": str(item["source"]),
            "time_col": str(item["time_col"]),
            "retention_days": int(item.get("retention_days", DEFAULT_RETENTION_DAYS)),
            "enable_archive": enable_archive,
            "archive_blob_prefix": archive_blob_prefix,
        })

    return validated

In [0]:
def setup_blob_credentials():
    """
    使用项目 env.py 中的 sap_touchpoint_blob_config 配置 Spark 访问归档 Blob 存储。
    该配置已经按环境（UAT/PROD）区分，确保归档到对应环境的存储账号。
    """
    blob_cfg = get_env_config("sap_touchpoint_blob_config")
    storage_name = blob_cfg["blob_storage_name"]
    storage_key = blob_cfg["blob_storage_key"]
    # 设置 Spark 访问 Azure Blob 所需的 account key
    spark.conf.set(
        f"fs.azure.account.key.{storage_name}.blob.core.windows.net",
        storage_key
    )

In [0]:
def read_source_df(item: dict) -> "DataFrame":
    """
    根据 source_type 从 Delta 路径或注册表读取源数据。

    参数:
        item: 单个归档组配置字典。

    返回:
        源数据的 Spark DataFrame。
    """
    # landing 一般为 Delta 路径（abfss://...）
    if item["source_type"] == "path":
        return spark.read.format("delta").load(item["source"])
    # clean / c-dataset 一般为注册表（catalog.database.table）
    return spark.table(item["source"])

In [0]:
def build_archive_path(archive_root: str, prefix: str, run_date: datetime) -> str:
    """
    构建本次运行的归档目标路径。

    参数:
        archive_root: Blob 归档根目录
        prefix: 该组数据的归档前缀（相对路径）
        run_date: 本次运行日期

    返回:
        形如 root/prefix/run_date=2025-07-20/ 的路径字符串
    """
    date_slug = run_date.strftime("%Y-%m-%d")
    return f"{archive_root.rstrip('/')}/{prefix}/"

def archive_df_to_blob(df, archive_path: str, run_date: datetime) -> int:
    """
    将 DataFrame 以 Delta 格式写入归档目录。

    参数:
        df: 待归档的 Spark DataFrame
        archive_path: 目标归档路径
        run_date: 本次运行日期

    返回:
        实际归档行数
    """
    date_slug = run_date.strftime("%Y-%m-%d")
    # 添加 run_date 列用于 replaceWhere 条件
    df_with_date = df.withColumn("_archive_run_date", F.lit(date_slug))
    # 使用 Delta overwrite + replaceWhere，按 run_date 分区安全重跑
    (df_with_date.write.format("delta").mode("append")
        # .option("replaceWhere", f"_archive_run_date = '{date_slug}'")
        .save(archive_path)
    )
    return df_with_date.count()


In [0]:
def delete_old_data(item: dict, cutoff_ts: datetime) -> None:
    """
    从源端删除 time_col < cutoff_ts 的行。

    参数:
        item: 单个归档组配置字典
        cutoff_ts:  cutoff 时间戳
    """
    time_col = item["time_col"]

    # 根据 source_type 选择 DeltaTable 构造方式
    if item["source_type"] == "table":
        delta_table = DeltaTable.forName(spark, item["source"])
    else:
        delta_table = DeltaTable.forPath(spark, item["source"])

    # 执行条件删除
    delta_table.delete(F.col(time_col) < F.lit(cutoff_ts))


In [0]:
def run_archive(config: list[dict], archive_root: str, run_date: datetime, dry_run: bool = False) -> list[dict]:
    """
    对每组配置按 retention_days 归档旧数据。

    每组处理流程：
      1. 计算 cutoff = run_date - retention_days
      2. 读取源 DataFrame
      3. 过滤出 time_col < cutoff 的行
      4. 如有数据，写入 Blob 归档路径
      5. 从源端删除相同行
      6. 记录结果

    参数:
        config: 校验后的配置列表
        archive_root: Blob 归档根目录
        run_date: 本次运行时间
        dry_run: True 时只统计匹配行数，不写入归档也不删除

    返回:
        每组执行结果列表
    """
    results = []

    for item in config:
        group_id = item["group_id"]
        time_col = item["time_col"]
        retention_days = item["retention_days"]
        # 计算 cutoff：早于该时间点的数据需要归档
        cutoff_ts = run_date - timedelta(days=retention_days)

        # 初始化结果字典
        result = {
            "group_id": group_id,
            "source": item["source"],
            "cutoff": cutoff_ts.isoformat(),
            "enable_archive": item["enable_archive"],
            "archived_rows": 0,
            "deleted_rows": 0,
            "error": None,
        }

        try:
            print(f"[{group_id}] cutoff: {cutoff_ts}, enable_archive: {item['enable_archive']}")

            # 读取源数据
            source_df = read_source_df(item)
            # 过滤出需要归档的旧数据
            old_df = source_df.filter(F.col(time_col) < F.lit(cutoff_ts))
            old_df.persist(StorageLevel.DISK_ONLY)

            try:
                # 统计待归档/删除行数
                old_count = old_df.count()
                print(f"[{group_id}] Rows to process: {old_count}")

                if dry_run:
                    print(f"[{group_id}] Dry run mode: skip archive and delete")
                    result["deleted_rows"] = old_count

                else:
                    if item["enable_archive"]:
                        # 构建并打印归档路径
                        archive_path = build_archive_path(archive_root, item["archive_blob_prefix"], run_date)
                        print(f"[{group_id}] Archive path: {archive_path}")

                        # 写入 Delta 归档目录
                        archive_df_to_blob(old_df, archive_path, run_date)
                        result["archived_rows"] = old_count

                    if old_count > 0:
                        # 从源端删除
                        delete_old_data(item, cutoff_ts)
                        print(f"[{group_id}] Deleted rows: {old_count}")
                        result["deleted_rows"] = old_count

            finally:
                # 释放缓存
                old_df.unpersist()
        except Exception as exc:
            # 捕获异常并记录，单组失败不影响其他组继续执行
            error_msg = f"{type(exc).__name__}: {exc}"
            print(f"[{group_id}] Error: {error_msg}")
            result["error"] = error_msg

        results.append(result)

    return results

In [0]:
def print_summary(results: list[dict], run_date: datetime):
    """
    打印本次归档执行的汇总信息。

    参数:
        results: run_archive 返回的结果列表
        run_date: 本次运行时间
    """
    print("=" * 60)
    print(f"Monthly Archive Summary - {run_date.isoformat()}")
    print("=" * 60)
    for r in results:
        status = "ERROR" if r["error"] else "OK"
        archive_info = f"archived={r['archived_rows']}" if r['enable_archive'] else "archive=skipped"
        print(
            f"[{status}] {r['group_id']}: source={r['source']}, "
            f"{archive_info}, deleted={r['deleted_rows']}, "
            f"cutoff={r['cutoff']}, error={r['error'] or 'none'}"
        )

In [0]:
# 从 Databricks Widget 读取运行参数
archive_config_json = dbutils.widgets.get("archive_config_json")
archive_root = dbutils.widgets.get("archive_root") or DEFAULT_ARCHIVE_ROOT

# 可选试运行模式：只统计匹配行数，不写入归档也不删除
try:
    dry_run = str(dbutils.widgets.get("dry_run")).strip().lower() == "true"
except Exception:
    dry_run = False

# 优先使用 Workflow 传入的触发时间，保证每月运行日期一致
try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except Exception:
    trigger_timestamp_ms = int(datetime.now(tz=timezone.utc).timestamp())

# 转换为 UTC datetime
run_date = datetime.fromtimestamp(trigger_timestamp_ms, tz=timezone.utc)

print(f"archive_root: {archive_root}")
print(f"run_date: {run_date}")
print(f"dry_run: {dry_run}")

# 解析配置并执行归档
config = parse_archive_config(archive_config_json)
results = run_archive(config, archive_root, run_date, dry_run=dry_run)
print_summary(results, run_date)